# A3 — Two measurers, one truth: agreement beyond luck

**From:** "they agree 88% of the time, nice"  **To:** deriving Cohen's kappa by hand and spotting the two classic traps (chance agreement, prevalence).

Setup, fully general: two measurers label the same items with yes/no. Two assay replicates calling binding/no-binding. Two doctors reading scans. And — where this is headed — **a human and an LLM judge labeling the same calls "task completed?"** The question is never "do they agree a lot?" but **"do they agree more than luck would produce?"**

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "rubric.yaml").exists())
import sys
sys.path.insert(0, str(ROOT / "pipeline"))
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
print("ready · repo:", ROOT.name)

n = 50
truth = rng.integers(0, 2, n)                       # ground truth, hidden from both
human = np.where(rng.random(n) < 0.92, truth, 1 - truth)   # decent rater: 8% slips
judge = np.where(rng.random(n) < 0.85, truth, 1 - truth)   # decent judge: 15% slips
raw = (human == judge).mean()
print(f"raw agreement: {raw:.2f}  - sounds impressive, no context yet")

## The broken judge that scores 90%
Imagine 90% of calls genuinely succeed, and a "judge" that just says **success every single time** — a constant function, measuring nothing. Watch its raw agreement:

In [ ]:
truth_skew = (rng.random(n) < 0.9).astype(int)
human_skew = np.where(rng.random(n) < 0.92, truth_skew, 1 - truth_skew)
lazy_judge = np.ones(n, dtype=int)
print(f"lazy judge raw agreement: {(human_skew == lazy_judge).mean():.2f}")
print("zero information, ~90% agreement. Raw agreement is broken as a metric.")

## Fixing it: subtract the luck
If the human says yes 92% of the time and the judge says yes 100% of the time, then *even with zero communication* they both say yes about 0.92×1.00 = 92% of the time, and both say no 0.08×0.00 = 0%. So **chance alone produces p_e ≈ 0.92 agreement.** General recipe:

p_e = P(both yes by luck) + P(both no by luck) = a₁·b₁ + a₀·b₀

where a₁, b₁ are each rater's yes-rates. **Cohen's kappa** then rescales observed agreement p_o against that floor:

κ = (p_o − p_e) / (1 − p_e)

Read it as: *of the agreement headroom above pure luck, what fraction did they capture?* κ=1 perfect, κ=0 exactly luck-level, κ<0 worse than luck.

In [ ]:
def kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    po = (a == b).mean()
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return (po - pe) / (1 - pe)

print(f"honest judge:  raw {(human == judge).mean():.2f}   kappa {kappa(human, judge):.2f}")
print(f"lazy judge:    raw {(human_skew == lazy_judge).mean():.2f}   kappa {kappa(human_skew, lazy_judge):.2f}")

The honest judge still captures over half the headroom above luck; the lazy judge collapses to **0.00** — exposed despite its higher raw score. That inversion (lower raw, higher kappa) is the entire reason kappa exists.

## The prevalence trap
Subtler: keep the *same* per-item slip rate, but make the classes imbalanced. **PREDICT:** does kappa rise, fall, or stay put as "yes" prevalence goes from 50% to 95%?

In [ ]:
prevs = np.linspace(0.5, 0.95, 10)
ks = []
for p in prevs:
    t = (rng.random(4000) < p).astype(int)
    h = np.where(rng.random(4000) < 0.92, t, 1 - t)
    j = np.where(rng.random(4000) < 0.85, t, 1 - t)
    ks.append(kappa(h, j))
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(prevs, ks, marker="o")
ax.set_xlabel("prevalence of 'yes'"); ax.set_ylabel("kappa")
ax.set_title("same raters, same slip rates - kappa falls as classes imbalance"); plt.show()

Same competence, lower kappa — because the luck floor p_e rises toward 1 when one class dominates, leaving little headroom to be better than. Two consequences for our project: (1) report prevalence alongside kappa, (2) when collecting the 40–60 labels, *stratify* so the binary dimension is not 95/5 — a balanced-ish label set makes kappa meaningful.

## Reading kappa, claiming honestly
The **Landis–Koch** convention: 0.41–0.60 moderate · **0.61–0.80 substantial** · 0.81+ almost perfect. House rule (locked in the spec): claim "substantial agreement" **only if the number AND its bootstrap CI (book A2!) sit inside 0.61–0.80.** Otherwise the phrase is "moderate, directional." The credibility move on top: show the items where the two measurers *disagreed* and discuss them — proof you studied the instrument instead of trusting it.

## Exercise — by hand, every step
Two label lists below (n=40, imbalanced on purpose). Compute p_o, the two yes-rates, p_e, and kappa with plain arithmetic — *then* check against the function.

In [ ]:
A = np.array([1,1,1,0,1,1,1,1,0,1,1,1,1,1,0,1,1,1,1,1,1,0,1,1,1,1,1,1,0,1,1,1,1,1,1,1,0,1,1,1])
B = np.array([1,1,1,1,1,1,1,1,0,1,1,0,1,1,0,1,1,1,1,1,1,1,1,1,1,0,1,1,0,1,1,1,1,1,1,1,1,1,1,1])
po = None      # YOUR TURN: fraction of positions where A==B
pe = None      # YOUR TURN: A.mean()*B.mean() + (1-A.mean())*(1-B.mean())
k  = None      # YOUR TURN: (po-pe)/(1-pe)
if k is not None:
    print(f"yours: {k:.3f}   function: {kappa(A, B):.3f}")
else:
    print("fill the three lines, then compare:", f"function says {kappa(A, B):.3f}")

## Self-check
1. Why is raw agreement a broken metric? (One devastating example.)
2. Derive p_e in words for two raters with yes-rates 0.9 and 0.8.
3. κ = 0.55 with CI [0.35, 0.72]: what may you claim?
4. Why will we stratify the calls Spike blind-labels?
5. **Gotcha:** your judge agrees with you at κ=0.78 — a colleague says "so it is right 78% of the time." Correct them.

<details><summary>Answers</summary>

1. A constant judge on imbalanced data scores ~prevalence agreement while measuring nothing (the lazy-judge demo).
2. Both-yes by luck 0.9×0.8=0.72, both-no 0.1×0.2=0.02 → p_e=0.74.
3. "Moderate, directional" — the interval dips well below 0.61, so "substantial" is not earned.
4. To keep the label classes balanced enough that the luck floor stays low and kappa retains headroom (and so every stress profile is represented).
5. Kappa is luck-adjusted *agreement with a human*, not accuracy against truth — and 0.78 is not a percentage of anything; it is the fraction of above-chance headroom captured.
</details>